<a href="https://colab.research.google.com/github/everestso/DeepLearningNotes/blob/main/d2026_11_11_SpotifyPlaylistExport.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Export a Spotify Playlist

In [ ]:
import base64
import hashlib
import secrets
from urllib.parse import urlencode
from IPython.display import display, Markdown

from google.colab import userdata

CLIENT_ID = userdata.get("SPOTIFY_CLIENT_ID").strip()
REDIRECT_URI = "http://127.0.0.1:8888/callback"

# Generate values used to secure this authorization session.
code_verifier = secrets.token_urlsafe(64)
code_challenge = base64.urlsafe_b64encode(
    hashlib.sha256(code_verifier.encode()).digest()
).decode().rstrip("=")

oauth_state = secrets.token_urlsafe(32)

params = {
    "client_id": CLIENT_ID,
    "response_type": "code",
    "redirect_uri": REDIRECT_URI,
    "scope": "playlist-read-private playlist-read-collaborative",
    "state": oauth_state,
    "code_challenge_method": "S256",
    "code_challenge": code_challenge,
}

auth_url = "https://accounts.spotify.com/authorize?" + urlencode(params)
display(Markdown(f"[Click here to authorize Spotify]({auth_url})"))

[Click here to authorize Spotify](https://accounts.spotify.com/authorize?client_id=562586e7f5e14f0ca9e37c02c811a596&response_type=code&redirect_uri=http%3A%2F%2F127.0.0.1%3A8888%2Fcallback&scope=playlist-read-private+playlist-read-collaborative&state=VQsNhwSACpOOCh6nLL7a200xurP4IUMWBa4Bnexz_rk&code_challenge_method=S256&code_challenge=QjcygcdaUox2wbwFafBSvbC9UmzF8VJkU-OfCSumnvA)

In [ ]:
import getpass
import requests
from urllib.parse import urlparse, parse_qs

callback_url = getpass.getpass(
    "Paste the entire callback URL here: "
).strip()

params = parse_qs(urlparse(callback_url).query)

if "error" in params:
    raise ValueError(f"Spotify authorization error: {params['error'][0]}")

if params.get("state", [""])[0] != oauth_state:
    raise ValueError(
        "Authorization session mismatch. Run the first cell again "
        "and authorize using its new link."
    )

if "code" not in params:
    raise ValueError("No authorization code found in the pasted URL.")

response = requests.post(
    "https://accounts.spotify.com/api/token",
    data={
        "grant_type": "authorization_code",
        "code": params["code"][0],
        "redirect_uri": REDIRECT_URI,
        "client_id": CLIENT_ID,
        "code_verifier": code_verifier,
    },
    timeout=30,
)

if not response.ok:
    error = response.json()
    raise RuntimeError(
        f"Spotify token exchange failed ({response.status_code}): "
        f"{error.get('error_description', error.get('error', 'Unknown error'))}"
    )

tokens = response.json()
access_token = tokens["access_token"]

print("Connected to Spotify successfully!")

Paste the entire callback URL here: ··········
Connected to Spotify successfully!


In [ ]:
import re
import pandas as pd
from google.colab import files

playlist_link = input("Paste your Spotify playlist link: ").strip()

# Accept a playlist link, Spotify URI, or playlist ID.
playlist_id = playlist_link.split("?")[0].rstrip("/")
playlist_id = playlist_id.split("/")[-1].split(":")[-1]

if not re.fullmatch(r"[A-Za-z0-9]{22}", playlist_id):
    raise ValueError("Please enter a valid Spotify playlist link or ID.")

headers = {"Authorization": f"Bearer {access_token}"}
url = (
    f"https://api.spotify.com/v1/playlists/{playlist_id}/items"
    "?limit=50"
)

rows = []

while url:
    response = requests.get(url, headers=headers, timeout=30)

    if response.status_code == 403:
        raise RuntimeError(
            "Access denied. You must own this playlist "
            "or be one of its collaborators."
        )

    if response.status_code == 401:
        raise RuntimeError(
            "Your session has expired. Repeat the authorization steps."
        )

    response.raise_for_status()
    page = response.json()

    for entry in page.get("items", []):
        track = entry.get("item") or entry.get("track")

        if not track or track.get("type") != "track":
            continue

        rows.append({
            "Song": track.get("name", ""),
            "Artist": ", ".join(
                artist["name"] for artist in track.get("artists", [])
            ),
            "Album": (track.get("album") or {}).get("name", ""),
        })

    url = page.get("next")

from IPython.display import display, Markdown

df = pd.DataFrame(rows)

if df.empty:
    print("No songs were returned.")
else:
    # Escape characters that could disrupt a Markdown table.
    table_df = df.map(
        lambda value: str(value)
        .replace("|", r"\|")
        .replace("\n", " ")
    )

    markdown_table = table_df.to_markdown(index=False)

    display(Markdown(markdown_table))
    print(f"\nTotal: {len(df)} songs")

    # Plain Markdown version for copying into other documents.
    print("\nCopyable Markdown:\n")
    print(markdown_table)

Paste your Spotify playlist link: https://open.spotify.com/playlist/3czYho9OLc1kSejYwxAvAW?si=_jIYTricTka3Ckf2VFAFzg


| Song                                                          | Artist                                                                     | Album                                               |
|:--------------------------------------------------------------|:---------------------------------------------------------------------------|:----------------------------------------------------|
| Anything for You                                              | Miami Sound Machine                                                        | Greatest Hits                                       |
| Boy Meets World Main Theme (From "Boy Meets World")           | Geek Music                                                                 | Boy Meets World Main Theme (From "Boy Meets World") |
| Kaleidoscope - From the New Broadway Musical “Hell’s Kitchen” | Alicia Keys, Maleah Joi Moon                                               | Hell’s Kitchen (Original Broadway Cast Recording)   |
| Lasso The Moon                                                | Ashe                                                                       | The Girl Of Your Dreams                             |
| Dime Cómo Quieres                                             | Christian Nodal, Ángela Aguilar                                            | Dime Cómo Quieres...                                |
| The Belle of St. Mark                                         | Sheila E.                                                                  | The Glamorous Life                                  |
| Black Prada Dress                                             | Ellie Goulding                                                             | Black Prada Dress                                   |
| Dress You Up                                                  | Madonna                                                                    | Like a Virgin                                       |
| Starchild                                                     | Teena Marie                                                                | Starchild (Expanded Edition)                        |
| Babyfather                                                    | Sade                                                                       | Soldier of Love                                     |
| Laapata                                                       | Sohail Sen, KK, Palak Muchhal, Anvita Dutt                                 | Ek Tha Tiger                                        |
| Kuch Parbat Hilaayein                                         | Arijit Singh                                                               | Poorna (Original Motion Picture Soundtrack)         |
| Tose Naina                                                    | Arijit Singh, Hanif Shaikh                                                 | Mickey Virus                                        |
| Full Jhol                                                     | Mika Singh, AKASA, Raj Hans                                                | Jackpot                                             |
| Kabhi Jo Baadal Barse (Feat. Maxi Priest)                     | Arijit Singh, Shaarib Toshi, Rishi Rich, Turaz, Azeem Shirazi, Maxi Priest | Jackpot                                             |
| Kabhi Jo Baadal Barse                                         | Arijit Singh, Shaarib Toshi, Turaz, Azeem Shirazi                          | Jackpot                                             |
| Ahtapotlar                                                    | Zakkum                                                                     | Her Geçen Yıl (2007 - 2013)                         |
| Nazlı Ay                                                      | Nazan Öncel                                                                | Bir Şarkı Tut                                       |
| Özledim                                                       | Esin İRİS                                                                  | Yine Mavi                                           |
| Boğaz'da Yangın Var                                           | Ayda                                                                       | Boğaz'da Yangın Var                                 |
| The Glamorous Life                                            | Sheila E.                                                                  | The Glamorous Life                                  |


Total: 21 songs

Copyable Markdown:

| Song                                                          | Artist                                                                     | Album                                               |
|:--------------------------------------------------------------|:---------------------------------------------------------------------------|:----------------------------------------------------|
| Anything for You                                              | Miami Sound Machine                                                        | Greatest Hits                                       |
| Boy Meets World Main Theme (From "Boy Meets World")           | Geek Music                                                                 | Boy Meets World Main Theme (From "Boy Meets World") |
| Kaleidoscope - From the New Broadway Musical “Hell’s Kitchen” | Alicia Keys, Maleah Joi Moon                                               | Hell’s Kitchen (Original Broadw

In [ ]:
from IPython.display import display, HTML

display(HTML(df.to_html(index=False, escape=True)))

Song,Artist,Album
Anything for You,Miami Sound Machine,Greatest Hits
"Boy Meets World Main Theme (From ""Boy Meets World"")",Geek Music,"Boy Meets World Main Theme (From ""Boy Meets World"")"
Kaleidoscope - From the New Broadway Musical “Hell’s Kitchen”,"Alicia Keys, Maleah Joi Moon",Hell’s Kitchen (Original Broadway Cast Recording)
Lasso The Moon,Ashe,The Girl Of Your Dreams
Dime Cómo Quieres,"Christian Nodal, Ángela Aguilar",Dime Cómo Quieres...
The Belle of St. Mark,Sheila E.,The Glamorous Life
Black Prada Dress,Ellie Goulding,Black Prada Dress
Dress You Up,Madonna,Like a Virgin
Starchild,Teena Marie,Starchild (Expanded Edition)
Babyfather,Sade,Soldier of Love


 Paste your Spotify playlist link: https://open.spotify.com/playlist/0YTaN5PTlHYPUmLZz5Yik6?si=Qa5MDJDrTjK504IG6nzeaA



Song
Artist
Album



Ya Te Olvidé
Yuridia
Para Mí


Mayores
Becky G, Bad Bunny
MALA SANTA


CHULA
Becky G
BARAJA BENDITA


Te Olvido (La La) (with Selena Gomez & Becky G)
benny blanco, Selena Gomez, Becky G
Hermoso


PATRONA
Becky G
BARAJA BENDITA


Burn
Luke Grimes
Luke Grimes


Lights Out
Lisa Marie Presley
To Whom It May Concern


Main Street U.S.A.
Grace Potter
Trespasser


Clothesline
Dylan Gossett
Clothesline


Qué Agonía
Yuridia, Ángela Aguilar
Pa' Luego Es Tarde


Que Se Vaya
Yuridia
Que Se Vaya


Sin Llorar
Yuridia
Sin Llorar


Me Hace Tanto Bien
Yuridia, Eden Muñoz
Pa' Luego Es Tarde


Waiting On the World to Change
John Mayer
Continuum


Brave
Sara Bareilles
The Blessed Unrest


Phone, Keys, Wallet
Lainey Wilson, John Mayer
Phone, Keys, Wallet


Waiting On the World to Change (feat. Ben Harper)
John Mayer, Ben Harper
The Village Sessions


Daughters
John Mayer
Heavier Things


Mad at Disney
salem ilese
Mad at Disney


Sexy Magic (from GTAVI: The Album)
CA7RIEL & Paco Amoroso, PinkPantheress, Fred again.., Etienne de Crécy , Grand Theft Auto VI
Sexy Magic (from GTAVI: The Album)


Last Thing You Need (from GTAVI: The Album)
Morgan Wallen, Grand Theft Auto VI
Last Thing You Need (from GTAVI: The Album)


Bright Lights, Big City (from GTAVI: The Album)
Keith Richards, Grand Theft Auto VI
Bright Lights, Big City (from GTAVI: The Album)


Macacoa 2000 (from GTAVI: The Album)
Rauw Alejandro, Grand Theft Auto VI
Macacoa 2000 (from GTAVI: The Album)


Dime Cómo Quieres
Christian Nodal, Ángela Aguilar
Dime Cómo Quieres...


La Llorona
Ángela Aguilar
PRIMERO SOY MEXICANA


La Tequilera
Ángela Aguilar
PRIMERO SOY MEXICANA


Paloma Negra
Ángela Aguilar
PRIMERO SOY MEXICANA


POR EL CONTRARIO (with Ángela Aguilar & Leonardo Aguilar) - Performance Video
Becky G, Ángela Aguilar, Leonardo Aguilar
POR EL CONTRARIO (with Ángela Aguilar & Leonardo Aguilar) [Performance Video]


POR EL CONTRARIO (with Ángela Aguilar & Leonardo Aguilar)
Becky G, Ángela Aguilar, Leonardo Aguilar
ESQUINAS


Jimmie Crack Corn
Burl Ives
Big Rock Candy Mountain


Jimmy Crack Corn
Muffin Songs
70 Best Kids Songs with Muffin Songs


Jimmy Crack Corn
Mike Frain
Newtown New World Old Ways


Blue Tail Fly
Burl Ives, The Andrews Sisters, Vic Schoen & His Orchestra
20th Century Masters: The Best of Burl Ives - The Millennium Collection


Come Find His Rest
Beckenhorst Singers, Craig Courtney, Dan Forrest
Beckenhorst Choral Music Spring 2023


Come Together
Aerosmith
Pandora's Box


Christine Sixteen
KISS
Love Gun


Sweet Hitchhiker
Sammy Hagar
Standing Hampton


Obsessed
Mariah Carey
Memoirs of an imperfect Angel


Jamie's Cryin' - 2015 Remaster
Van Halen
Van Halen (Remastered)


Illusion
Dua Lipa
Radical Optimism


Levitating
Dua Lipa
Future Nostalgia


Kinda Love It
Lainey Wilson
Kinda Love It


Casually Complicated
Mercer Henderson
Casually Complicated


Memories
Charlotte Cardin
Memories


Girl's A Goner
Tigirlily Gold
Country & Midwestern


HOLD THE LIGHT
Parisalexa
HOLD THE LIGHT


Somewhere Over Laredo
Lainey Wilson
Whirlwind (Deluxe)


When I Dream
Choi Kyung Sik
Beakya 3.98 OST (Soundtrack)


Moon River (From Breakfast at Tiffany's) [Remastered]
Audrey Hepburn
Moon River (Breakfast at Tiffany's) [Remastered]


My Own Home (The Jungle Book Theme)
Darleen Carr
The Jungle Book


On My Own (From "Les misérables") [Live]
Lea Salonga
The Story of My Life: Lea Salonga Live from Manila


I Dreamed a Dream (From "Les misérables") [Live]
Lea Salonga
The Story of My Life: Lea Salonga Live from Manila


Lea Salonga Overture: Sun and Moon / A Whole New World / I Dreamed a Dream / On My Own / Reflection (Live)
BYU Chamber Orchestra, Kory Katseanes
The Story of My Life: Lea Salonga Live from Manila


Help Me Make It Through the Night
Kris Kristofferson
Kristofferson


Total: 54 songs

Copyable Markdown:

| Song                                                                                                       | Artist                                                                                       | Album                                                                         |
|:-----------------------------------------------------------------------------------------------------------|:---------------------------------------------------------------------------------------------|:------------------------------------------------------------------------------|
| Ya Te Olvidé                                                                                               | Yuridia                                                                                      | Para Mí                                                                       |
| Mayores                                                                                                    | Becky G, Bad Bunny                                                                           | MALA SANTA                                                                    |
| CHULA                                                                                                      | Becky G                                                                                      | BARAJA BENDITA                                                                |
| Te Olvido (La La) (with Selena Gomez & Becky G)                                                            | benny blanco, Selena Gomez, Becky G                                                          | Hermoso                                                                       |
| PATRONA                                                                                                    | Becky G                                                                                      | BARAJA BENDITA                                                                |
| Burn                                                                                                       | Luke Grimes                                                                                  | Luke Grimes                                                                   |
| Lights Out                                                                                                 | Lisa Marie Presley                                                                           | To Whom It May Concern                                                        |
| Main Street U.S.A.                                                                                         | Grace Potter                                                                                 | Trespasser                                                                    |
| Clothesline                                                                                                | Dylan Gossett                                                                                | Clothesline                                                                   |
| Qué Agonía                                                                                                 | Yuridia, Ángela Aguilar                                                                      | Pa' Luego Es Tarde                                                            |
| Que Se Vaya                                                                                                | Yuridia                                                                                      | Que Se Vaya                                                                   |
| Sin Llorar                                                                                                 | Yuridia                                                                                      | Sin Llorar                                                                    |
| Me Hace Tanto Bien                                                                                         | Yuridia, Eden Muñoz                                                                          | Pa' Luego Es Tarde                                                            |
| Waiting On the World to Change                                                                             | John Mayer                                                                                   | Continuum                                                                     |
| Brave                                                                                                      | Sara Bareilles                                                                               | The Blessed Unrest                                                            |
| Phone, Keys, Wallet                                                                                        | Lainey Wilson, John Mayer                                                                    | Phone, Keys, Wallet                                                           |
| Waiting On the World to Change (feat. Ben Harper)                                                          | John Mayer, Ben Harper                                                                       | The Village Sessions                                                          |
| Daughters                                                                                                  | John Mayer                                                                                   | Heavier Things                                                                |
| Mad at Disney                                                                                              | salem ilese                                                                                  | Mad at Disney                                                                 |
| Sexy Magic (from GTAVI: The Album)                                                                         | CA7RIEL & Paco Amoroso, PinkPantheress, Fred again.., Etienne de Crécy , Grand Theft Auto VI | Sexy Magic (from GTAVI: The Album)                                            |
| Last Thing You Need (from GTAVI: The Album)                                                                | Morgan Wallen, Grand Theft Auto VI                                                           | Last Thing You Need (from GTAVI: The Album)                                   |
| Bright Lights, Big City (from GTAVI: The Album)                                                            | Keith Richards, Grand Theft Auto VI                                                          | Bright Lights, Big City (from GTAVI: The Album)                               |
| Macacoa 2000 (from GTAVI: The Album)                                                                       | Rauw Alejandro, Grand Theft Auto VI                                                          | Macacoa 2000 (from GTAVI: The Album)                                          |
| Dime Cómo Quieres                                                                                          | Christian Nodal, Ángela Aguilar                                                              | Dime Cómo Quieres...                                                          |
| La Llorona                                                                                                 | Ángela Aguilar                                                                               | PRIMERO SOY MEXICANA                                                          |
| La Tequilera                                                                                               | Ángela Aguilar                                                                               | PRIMERO SOY MEXICANA                                                          |
| Paloma Negra                                                                                               | Ángela Aguilar                                                                               | PRIMERO SOY MEXICANA                                                          |
| POR EL CONTRARIO (with Ángela Aguilar & Leonardo Aguilar) - Performance Video                              | Becky G, Ángela Aguilar, Leonardo Aguilar                                                    | POR EL CONTRARIO (with Ángela Aguilar & Leonardo Aguilar) [Performance Video] |
| POR EL CONTRARIO (with Ángela Aguilar & Leonardo Aguilar)                                                  | Becky G, Ángela Aguilar, Leonardo Aguilar                                                    | ESQUINAS                                                                      |
| Jimmie Crack Corn                                                                                          | Burl Ives                                                                                    | Big Rock Candy Mountain                                                       |
| Jimmy Crack Corn                                                                                           | Muffin Songs                                                                                 | 70 Best Kids Songs with Muffin Songs                                          |
| Jimmy Crack Corn                                                                                           | Mike Frain                                                                                   | Newtown New World Old Ways                                                    |
| Blue Tail Fly                                                                                              | Burl Ives, The Andrews Sisters, Vic Schoen & His Orchestra                                   | 20th Century Masters: The Best of Burl Ives - The Millennium Collection       |
| Come Find His Rest                                                                                         | Beckenhorst Singers, Craig Courtney, Dan Forrest                                             | Beckenhorst Choral Music Spring 2023                                          |
| Come Together                                                                                              | Aerosmith                                                                                    | Pandora's Box                                                                 |
| Christine Sixteen                                                                                          | KISS                                                                                         | Love Gun                                                                      |
| Sweet Hitchhiker                                                                                           | Sammy Hagar                                                                                  | Standing Hampton                                                              |
| Obsessed                                                                                                   | Mariah Carey                                                                                 | Memoirs of an imperfect Angel                                                 |
| Jamie's Cryin' - 2015 Remaster                                                                             | Van Halen                                                                                    | Van Halen (Remastered)                                                        |
| Illusion                                                                                                   | Dua Lipa                                                                                     | Radical Optimism                                                              |
| Levitating                                                                                                 | Dua Lipa                                                                                     | Future Nostalgia                                                              |
| Kinda Love It                                                                                              | Lainey Wilson                                                                                | Kinda Love It                                                                 |
| Casually Complicated                                                                                       | Mercer Henderson                                                                             | Casually Complicated                                                          |
| Memories                                                                                                   | Charlotte Cardin                                                                             | Memories                                                                      |
| Girl's A Goner                                                                                             | Tigirlily Gold                                                                               | Country & Midwestern                                                          |
| HOLD THE LIGHT                                                                                             | Parisalexa                                                                                   | HOLD THE LIGHT                                                                |
| Somewhere Over Laredo                                                                                      | Lainey Wilson                                                                                | Whirlwind (Deluxe)                                                            |
| When I Dream                                                                                               | Choi Kyung Sik                                                                               | Beakya 3.98 OST (Soundtrack)                                                  |
| Moon River (From Breakfast at Tiffany's) [Remastered]                                                      | Audrey Hepburn                                                                               | Moon River (Breakfast at Tiffany's) [Remastered]                              |
| My Own Home (The Jungle Book Theme)                                                                        | Darleen Carr                                                                                 | The Jungle Book                                                               |
| On My Own (From "Les misérables") [Live]                                                                   | Lea Salonga                                                                                  | The Story of My Life: Lea Salonga Live from Manila                            |
| I Dreamed a Dream (From "Les misérables") [Live]                                                           | Lea Salonga                                                                                  | The Story of My Life: Lea Salonga Live from Manila                            |
| Lea Salonga Overture: Sun and Moon / A Whole New World / I Dreamed a Dream / On My Own / Reflection (Live) | BYU Chamber Orchestra, Kory Katseanes                                                        | The Story of My Life: Lea Salonga Live from Manila                            |
| Help Me Make It Through the Night                                                                          | Kris Kristofferson                                                                           | Kristofferson                                                                 |
